# DER Inventory

**Infrastructure Step 1.** Build critical-facility, critical-load assignment, and `der_inventory.parquet` artifacts from evidence-backed public facility records.

Two layers:
- **Layer 1** — one DER row per Marshfield Hazard Management Plan-evidenced public backup-power facility (`evidence_anchored_mhmp`).
- **Layer 2** — REopt-sized PV/battery/generator capacity can be refreshed later once load profiles exist.

> **Stage Contract**
>
> Requires: base network asset registry and location-local critical_facilities.geojson
>
> Produces: critical load assignments, DER inventory seed, reviewed critical facility artifact
>
> Next: 02_load_profiles.ipynb

## Runtime

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

location_root = Path("../..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from power.notebook import load_runtime

runtime = load_runtime(location_root)
location_root = runtime.location_root
location_name = runtime.location_name
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths
grid = runtime.grid


augmented_artifacts: /home/grahamhults/projects/Flood-RM/locations/marshfield/data/static/power_grid/smart_ds_compat


## Layer 1 — Evidence-Anchored MHMP Facilities

Seed one DER row per MHMP-backed facility. This cell also writes the supporting `critical_facilities.parquet` and `critical_load_assignments.parquet` artifacts that downstream load-profile and ONM steps consume.

In [2]:
from power.resilience import (
    CRITICAL_LOAD_ASSIGNMENTS_SCHEMA_VERSION,
    build_critical_load_assignments,
    critical_load_assignments_pyarrow_schema,
    validate_critical_load_assignments,
)
from power.resilience import (
    SCHEMA_VERSION as DER_INVENTORY_SCHEMA_VERSION,
    build_inventory,
    der_inventory_pyarrow_schema,
    validate_der_assignment_completeness,
)
from power.resilience import (
    CRITICAL_FACILITY_SCHEMA_VERSION,
    load_critical_facilities,
    load_bus_electrical_metadata,
    write_critical_facilities_artifact,
)
import pyarrow as pa
import pyarrow.parquet as pq

assets_path = grid["augmented_artifacts"] / "assets.parquet"
control_units_path = grid["augmented_artifacts"] / "control_units.parquet"
critical_facilities_source = location_root / config["grid"]["critical_facilities"]["source"]
critical_facilities_path = grid["augmented_artifacts"] / "critical_facilities.parquet"
critical_load_assignments_path = grid["augmented_artifacts"] / "critical_load_assignments.parquet"
der_inventory_path = grid["augmented_artifacts"] / "der_inventory.parquet"

if not assets_path.exists() or not control_units_path.exists():
    missing = [p.relative_to(repo_root) for p in (assets_path, control_units_path) if not p.exists()]
    raise FileNotFoundError(
        "Run 01_base_network.ipynb first; missing Stage A1 artifacts: "
        + ", ".join(str(p) for p in missing)
    )

assets = pd.read_parquet(assets_path)
control_units = pd.read_parquet(control_units_path)
loads = pd.read_csv(grid["asset_registry"] / "loads.csv")
buses = pd.read_csv(grid["asset_registry"] / "buses.csv")

critical_facility_gdf = load_critical_facilities(critical_facilities_source, location_name=paths["location_name"])
write_critical_facilities_artifact(critical_facility_gdf, critical_facilities_path)
critical_facility_records = critical_facility_gdf.drop(columns=["geometry"], errors="ignore").to_dict("records")

critical_load_assignment_rows = build_critical_load_assignments(
    critical_facility_records,
    asset_rows=assets.to_dict("records"),
    control_unit_rows=control_units.to_dict("records"),
    load_bus_electrical_metadata=load_bus_electrical_metadata(loads),
    sandbox_id=paths["location_name"],
    assign_nearest_when_outside_radius=True,
)
validate_critical_load_assignments(
    critical_load_assignment_rows,
    facility_ids={row["facility_id"] for row in critical_facility_records},
    asset_ids=set(assets["asset_id"].dropna().astype(str)),
    control_unit_ids=set(control_units["control_unit_id"].dropna().astype(str)),
)

assignment_schema = critical_load_assignments_pyarrow_schema()
assignment_columns = {
    field.name: [row.get(field.name) for row in critical_load_assignment_rows]
    for field in assignment_schema
}
pq.write_table(pa.table(assignment_columns, schema=assignment_schema), critical_load_assignments_path)

layer1_rows = build_inventory(
    critical_facility_records,
    critical_load_assignments=critical_load_assignment_rows,
    sandbox_id=paths["location_name"],
)
validate_der_assignment_completeness(
    layer1_rows,
    valid_buses=set(buses["bus"].dropna().astype(str)),
)

der_schema = der_inventory_pyarrow_schema()
der_columns = {field.name: [row.get(field.name) for row in layer1_rows] for field in der_schema}
pq.write_table(pa.table(der_columns, schema=der_schema), der_inventory_path)

der_inventory = pd.DataFrame(layer1_rows)
print(f"Critical facilities: {len(critical_facility_records)} ({CRITICAL_FACILITY_SCHEMA_VERSION})")
print(f"Critical load assignments: {len(critical_load_assignment_rows)} ({CRITICAL_LOAD_ASSIGNMENTS_SCHEMA_VERSION})")
print(f"Layer 1 DER rows: {len(layer1_rows)} ({DER_INVENTORY_SCHEMA_VERSION})")
display(der_inventory[["facility_id", "load_asset_id", "bus", "assignment_status", "placement_rule", "genset_kw"]].head(20))

Critical facilities: 16 (stage_b_critical_facilities.v0.2)
Critical load assignments: 16 (stage_b_critical_load_assignments.v0.2)
Layer 1 DER rows: 15 (stage_b_der_inventory.v0.1)


,facility_id,load_asset_id,bus,assignment_status,placement_rule,genset_kw
0,marshfield:critical_facility:town_hall_870_mor...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_023__470f522...,assigned,evidence_anchored_mhmp,None
1,marshfield:critical_facility:police_eoc_1639_o...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_029__d380736...,assigned,evidence_anchored_mhmp,None
2,marshfield:critical_facility:central_fire_stat...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_017__f63c710...,assigned,evidence_anchored_mhmp,None
3,marshfield:critical_facility:dpw_building_965_...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_007__0e07868...,assigned,evidence_anchored_mhmp,None
4,marshfield:critical_facility:governor_winslow_...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_033__bee5b99...,assigned,evidence_anchored_mhmp,None
5,marshfield:critical_facility:furnace_brook_mid...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_012__15dea0e...,assigned,evidence_anchored_mhmp,None
6,marshfield:critical_facility:south_river_schoo...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_017__8b150b0...,assigned,evidence_anchored_mhmp,None
7,marshfield:critical_facility:daniel_webster_sc...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_029__ac65698...,assigned,evidence_anchored_mhmp,None
8,marshfield:critical_facility:marshfield_high_s...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_012__666819c...,assigned,evidence_anchored_mhmp,None
9,marshfield:critical_facility:martinson_element...,marshfield:asset:load_buses:marshfield_shift_s...,marshfield_shift_synthetic_region_012__dcf5068...,assigned,evidence_anchored_mhmp,None


## Layer 2 — REopt Sizing

Layer 2 sizing depends on `load_profile_assignments.parquet`, which is built in `02_load_profiles.ipynb`. Keep this notebook cache-free and write Layer 1 rows only; run the REopt sizing helper after load profiles are refreshed when capacity sizing is intentional.

In [3]:
from power.resilience import CachedReoptResultsClient, size_der

print("Layer 2 REopt replay is intentionally deferred until after 02_load_profiles.ipynb.")
print("Use size_der(...) with a live or cached client when refreshing capacities.")

Layer 2 REopt replay is intentionally deferred until after 02_load_profiles.ipynb.
Use run_layer_2_reopt_sizing(...) with a live or cached client when refreshing capacities.


## Write Artifact

In [4]:
print(f"Wrote {len(der_inventory):,} rows to {der_inventory_path.relative_to(repo_root)}")
print(f"Critical facilities: {critical_facilities_path.relative_to(repo_root)}")
print(f"Critical load assignments: {critical_load_assignments_path.relative_to(repo_root)}")

Wrote 15 rows to locations/marshfield/data/static/power_grid/smart_ds_compat/der_inventory.parquet
Critical facilities: locations/marshfield/data/static/power_grid/smart_ds_compat/critical_facilities.parquet
Critical load assignments: locations/marshfield/data/static/power_grid/smart_ds_compat/critical_load_assignments.parquet
